# Módulo 36: Operaciones en tablas

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m36-operaciones-en-tablas-solucion.sql](m36-operaciones-en-tablas-solucion.sql). No es una tarea entregada del programa.

En el Módulo 35 solo leímos datos. Ahora toca **crearlos y cambiarlos**: definir tablas con sus tipos y restricciones, insertar filas, modificarlas, borrarlas y alterar la estructura de una tabla que ya tiene datos. Como antes, ejecutamos todo en SQLite y mostramos la forma T-SQL cuando la sintaxis difiere.

1. DDL y DML
2. Tipos de datos y afinidad en SQLite
3. Restricciones y por qué nombrarlas
4. `INSERT` de varias filas
5. `ALTER TABLE`: agregar y renombrar
6. Cambiar una columna o una restricción: reconstruir la tabla
7. `UPDATE` y `DELETE`
8. `DELETE`, `TRUNCATE` y `DROP`
9. `CASE`, `CAST` y concatenación
10. Errores frecuentes
11. Ejercicio

## 1. DDL y DML

Las sentencias SQL se agrupan según lo que modifican:

| Grupo | Significado | Sentencias | Actúa sobre |
| --- | --- | --- | --- |
| **DDL** | *Data Definition Language* | `CREATE`, `ALTER`, `DROP`, `TRUNCATE` | La estructura: tablas, columnas, restricciones |
| **DML** | *Data Manipulation Language* | `INSERT`, `UPDATE`, `DELETE` (y `SELECT`) | Las filas que contiene la tabla |

La distinción importa en la práctica: un error de DDL cambia la forma de la tabla para todas las consultas que dependen de ella, mientras que uno de DML altera datos concretos.

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")


def q(sql):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)

## 2. Tipos de datos y afinidad en SQLite

En SQL Server cada columna tiene un tipo estricto: `INT`, `DECIMAL(10, 2)`, `VARCHAR(50)`, `NVARCHAR(MAX)`, `DATE`, `DATETIME`, `CHAR(1)`... Si se inserta un valor incompatible, la sentencia falla.

SQLite es más flexible: cada columna tiene una **afinidad** que solo indica la conversión preferida, y cada valor guarda su propio tipo. La afinidad se deduce del nombre del tipo declarado:

| Tipo declarado (ejemplos) | Afinidad en SQLite |
| --- | --- |
| `INT`, `INTEGER`, `BIGINT` | INTEGER |
| `VARCHAR(50)`, `NVARCHAR(100)`, `CHAR(1)`, `TEXT` | TEXT |
| `REAL`, `FLOAT`, `DOUBLE` | REAL |
| `DECIMAL(10, 2)`, `NUMERIC`, `DATE`, `DATETIME` | NUMERIC |
| Sin tipo, o `BLOB` | BLOB (sin conversión) |

SQLite no tiene un tipo de fecha propio: las fechas se guardan como texto ISO (`'2024-03-15'`) y se manipulan con funciones como `date` o `strftime`.

In [2]:
conn.execute("CREATE TABLE Demo (entero INTEGER, texto VARCHAR(10), decimal DECIMAL(10, 2))")
conn.execute("INSERT INTO Demo VALUES ('42', 42, '19.90')")      # tipos "cruzados"
conn.execute("INSERT INTO Demo VALUES ('hola', 'una cadena larguísima', 'n/a')")

# typeof() muestra el tipo con el que quedó guardado cada valor
q("SELECT entero, typeof(entero), texto, typeof(texto), decimal, typeof(decimal) FROM Demo")

,entero,typeof(entero),texto,typeof(texto),decimal,typeof(decimal)
0,42,integer,42,text,19.9,real
1,hola,text,una cadena larguísima,text,n/a,text


En la primera fila, `'42'` se convirtió a entero y `42` a texto, porque la conversión era posible. En la segunda, `'hola'` se guardó como texto en una columna `INTEGER`, y `VARCHAR(10)` no impidió guardar un texto más largo: SQLite **no aplica la longitud declarada**. Para validar datos hay que usar restricciones.

## 3. Restricciones y por qué nombrarlas

Una **restricción** (*constraint*) es una regla que la base hace cumplir en cada `INSERT` y `UPDATE`:

| Restricción | Garantiza |
| --- | --- |
| `PRIMARY KEY` | Valor único que identifica la fila |
| `NOT NULL` | Que la columna siempre tenga valor |
| `UNIQUE` | Que no haya valores repetidos |
| `CHECK (condición)` | Que la condición se cumpla |
| `DEFAULT valor` | Un valor cuando no se indica ninguno (no rechaza filas) |

Cada restricción puede llevar un nombre con `CONSTRAINT nombre ...`. Si no se le da, SQL Server inventa uno como `PK__Envio__3A2F1B7C...`, distinto en cada instalación. Nombrarla tiene dos ventajas:

1. Los mensajes de error indican **qué regla** se violó.
2. Para eliminar o modificar la restricción en T-SQL (`ALTER TABLE ... DROP CONSTRAINT nombre`) hay que conocer su nombre; con uno explícito, el mismo script funciona en cualquier servidor.

In [3]:
conn.execute("""
CREATE TABLE Envio (
    NumeroEnvio  INTEGER NOT NULL,
    FechaEnvio   TEXT    NOT NULL,
    Cliente      TEXT    NOT NULL,
    Express      TEXT    DEFAULT 'N',
    PesoKg       REAL,
    Costo        DECIMAL(10, 2),
    CONSTRAINT PK_Envio PRIMARY KEY (NumeroEnvio),
    CONSTRAINT CK_Envio_Express CHECK (Express IN ('S', 'N')),
    CONSTRAINT CK_Envio_Costo   CHECK (Costo >= 0)
)
""")

# Cada intento viola una restricción distinta
intentos = [
    "INSERT INTO Envio VALUES (1, '2024-03-01', 'Ana', 'X', 2.0, 150)",
    "INSERT INTO Envio VALUES (2, '2024-03-01', 'Luis', 'N', 1.0, -10)",
    "INSERT INTO Envio VALUES (3, '2024-03-01', NULL, 'N', 1.0, 80)",
]
for sql in intentos:
    try:
        conn.execute(sql)
    except sqlite3.IntegrityError as error:
        print("Rechazado:", error)

Rechazado: CHECK constraint failed: CK_Envio_Express
Rechazado: CHECK constraint failed: CK_Envio_Costo
Rechazado: NOT NULL constraint failed: Envio.Cliente


Los dos `CHECK` aparecen con su nombre en el mensaje. En SQL Server, la tabla equivalente se escribiría así:

```sql
CREATE TABLE dbo.Envio (
    NumeroEnvio  INT            NOT NULL,
    FechaEnvio   DATE           NOT NULL,
    Cliente      NVARCHAR(100)  NOT NULL,
    Express      CHAR(1)        CONSTRAINT DF_Envio_Express DEFAULT 'N',
    PesoKg       DECIMAL(6, 2),
    Costo        DECIMAL(10, 2),
    CONSTRAINT PK_Envio PRIMARY KEY (NumeroEnvio),
    CONSTRAINT CK_Envio_Express CHECK (Express IN ('S', 'N')),
    CONSTRAINT CK_Envio_Costo   CHECK (Costo >= 0)
);
```

## 4. `INSERT` de varias filas

Una sola sentencia `INSERT` puede añadir muchas filas separando cada tupla por comas. Conviene **listar las columnas** de forma explícita: así la sentencia no depende del orden en que se crearon, y las columnas omitidas toman su `DEFAULT` (o `NULL`).

In [4]:
conn.execute("""
INSERT INTO Envio (NumeroEnvio, FechaEnvio, Cliente, Express, PesoKg, Costo)
VALUES
    (1001, '2024-03-01', 'Ana Ruiz',     'S', 2.5, 180.00),
    (1002, '2024-03-01', 'Luis Mora',    'N', 1.2,  95.50),
    (1003, '2024-03-02', 'Marta Díaz',   'S', 4.0, 240.00),
    (1004, '2024-03-03', 'Pedro Sáenz',  'N', 0.8,  60.00),
    (1005, '2024-03-03', 'Sofía Lara',   'N', 3.1, 120.00)
""")
# Express se omite: toma el valor por defecto 'N'
conn.execute("INSERT INTO Envio (NumeroEnvio, FechaEnvio, Cliente, PesoKg, Costo) VALUES (1006, '2024-03-04', 'Raúl Pinto', 1.0, 70.00)")
conn.commit()

q("SELECT * FROM Envio")

,NumeroEnvio,FechaEnvio,Cliente,Express,PesoKg,Costo
0,1001,2024-03-01,Ana Ruiz,S,2.5,180.0
1,1002,2024-03-01,Luis Mora,N,1.2,95.5
2,1003,2024-03-02,Marta Díaz,S,4.0,240.0
3,1004,2024-03-03,Pedro Sáenz,N,0.8,60.0
4,1005,2024-03-03,Sofía Lara,N,3.1,120.0
5,1006,2024-03-04,Raúl Pinto,N,1.0,70.0


## 5. `ALTER TABLE`: agregar y renombrar

Para cambios simples de estructura, SQLite y SQL Server se parecen, salvo al renombrar:

| Cambio | SQLite | T-SQL |
| --- | --- | --- |
| Agregar columna | `ALTER TABLE t ADD COLUMN c TIPO` | `ALTER TABLE t ADD c TIPO` |
| Renombrar tabla | `ALTER TABLE t RENAME TO nuevo` | `EXEC sp_rename 'dbo.t', 'nuevo'` |
| Renombrar columna | `ALTER TABLE t RENAME COLUMN c TO nuevo` | `EXEC sp_rename 'dbo.t.c', 'nuevo', 'COLUMN'` |

La columna nueva se rellena con `NULL` (o con su `DEFAULT`) en las filas existentes.

In [5]:
conn.execute("ALTER TABLE Envio ADD COLUMN FechaEntrega TEXT")
conn.execute("ALTER TABLE Envio RENAME COLUMN Express TO EsExpress")
conn.execute("ALTER TABLE Envio RENAME TO Despacho")

q("SELECT NumeroEnvio, Cliente, EsExpress, FechaEntrega FROM Despacho LIMIT 3")

,NumeroEnvio,Cliente,EsExpress,FechaEntrega
0,1001,Ana Ruiz,S,None
1,1002,Luis Mora,N,None
2,1003,Marta Díaz,S,None


Renombrar la tabla **no renombra sus restricciones**: siguen llamándose `PK_Envio`, `CK_Envio_Express`... SQLite guarda la definición original, y el `RENAME COLUMN` actualizó también la referencia dentro del `CHECK`:

In [6]:
print(conn.execute("SELECT sql FROM sqlite_master WHERE name = 'Despacho'").fetchone()[0])

CREATE TABLE "Despacho" (
    NumeroEnvio  INTEGER NOT NULL,
    FechaEnvio   TEXT    NOT NULL,
    Cliente      TEXT    NOT NULL,
    EsExpress      TEXT    DEFAULT 'N',
    PesoKg       REAL,
    Costo        DECIMAL(10, 2), FechaEntrega TEXT,
    CONSTRAINT PK_Envio PRIMARY KEY (NumeroEnvio),
    CONSTRAINT CK_Envio_Express CHECK (EsExpress IN ('S', 'N')),
    CONSTRAINT CK_Envio_Costo   CHECK (Costo >= 0)
)


## 6. Cambiar una columna o una restricción: reconstruir la tabla

SQL Server permite cambiar el tipo de una columna y quitar o añadir restricciones sobre la tabla existente:

```sql
ALTER TABLE dbo.Despacho DROP CONSTRAINT PK_Envio;
ALTER TABLE dbo.Despacho ALTER COLUMN NumeroEnvio VARCHAR(12) NOT NULL;
UPDATE dbo.Despacho
SET NumeroEnvio = CASE WHEN EsExpress = 'S' THEN 'EX-' ELSE 'ST-' END
                  + CAST(NumeroEnvio AS VARCHAR(10));
ALTER TABLE dbo.Despacho ADD CONSTRAINT PK_Despacho PRIMARY KEY (NumeroEnvio);
```

SQLite **no tiene** `ALTER COLUMN` ni `DROP/ADD CONSTRAINT`. El procedimiento recomendado es reconstruir la tabla dentro de una transacción:

1. Crear una tabla nueva con la estructura deseada.
2. Copiar los datos con `INSERT ... SELECT`, transformándolos si hace falta.
3. Borrar la tabla original.
4. Renombrar la nueva con el nombre original.

In [7]:
conn.executescript("""
BEGIN;

CREATE TABLE Despacho_nuevo (
    NumeroEnvio  TEXT NOT NULL,             -- antes INTEGER
    FechaEnvio   TEXT NOT NULL,
    Cliente      TEXT NOT NULL,
    EsExpress    TEXT DEFAULT 'N',
    PesoKg       REAL,
    Costo        DECIMAL(10, 2),
    FechaEntrega TEXT,
    CONSTRAINT PK_Despacho PRIMARY KEY (NumeroEnvio),
    CONSTRAINT CK_Despacho_Express CHECK (EsExpress IN ('S', 'N')),
    CONSTRAINT CK_Despacho_Costo   CHECK (Costo >= 0)
);

INSERT INTO Despacho_nuevo
SELECT CASE WHEN EsExpress = 'S' THEN 'EX-' ELSE 'ST-' END || CAST(NumeroEnvio AS TEXT),
       FechaEnvio, Cliente, EsExpress, PesoKg, Costo, FechaEntrega
FROM Despacho;

DROP TABLE Despacho;
ALTER TABLE Despacho_nuevo RENAME TO Despacho;

COMMIT;
""")

q("SELECT NumeroEnvio, typeof(NumeroEnvio) AS tipo, Cliente, EsExpress FROM Despacho")

,NumeroEnvio,tipo,Cliente,EsExpress
0,EX-1001,text,Ana Ruiz,S
1,ST-1002,text,Luis Mora,N
2,EX-1003,text,Marta Díaz,S
3,ST-1004,text,Pedro Sáenz,N
4,ST-1005,text,Sofía Lara,N
5,ST-1006,text,Raúl Pinto,N


Si algo falla antes del `COMMIT`, la transacción se puede deshacer y la tabla original queda intacta. Por eso los cuatro pasos van juntos.

## 7. `UPDATE` y `DELETE`

`UPDATE` cambia valores de filas existentes y `DELETE` elimina filas. Ambas afectan **a todas las filas que cumplan el `WHERE`**; si no hay `WHERE`, afectan a toda la tabla.

Una práctica segura es escribir primero un `SELECT` con el mismo `WHERE`, revisar qué filas devuelve y solo después cambiarlo por `UPDATE` o `DELETE`.

In [8]:
# Primero: ¿qué filas se van a tocar?
display(q("SELECT NumeroEnvio, Cliente, Costo FROM Despacho WHERE EsExpress = 'S'"))

cur = conn.execute("UPDATE Despacho SET Costo = Costo * 1.10 WHERE EsExpress = 'S'")
print("Filas actualizadas:", cur.rowcount)

cur = conn.execute("UPDATE Despacho SET FechaEntrega = '2024-03-06' WHERE NumeroEnvio = 'ST-1002'")
print("Filas actualizadas:", cur.rowcount)

cur = conn.execute("DELETE FROM Despacho WHERE PesoKg < 1")
print("Filas eliminadas:", cur.rowcount)
conn.commit()

q("SELECT NumeroEnvio, Cliente, round(Costo, 2) AS Costo, FechaEntrega FROM Despacho")

,NumeroEnvio,Cliente,Costo
0,EX-1001,Ana Ruiz,180
1,EX-1003,Marta Díaz,240


Filas actualizadas: 2
Filas actualizadas: 1
Filas eliminadas: 1


,NumeroEnvio,Cliente,Costo,FechaEntrega
0,EX-1001,Ana Ruiz,198.0,None
1,ST-1002,Luis Mora,95.5,2024-03-06
2,EX-1003,Marta Díaz,264.0,None
3,ST-1005,Sofía Lara,120.0,None
4,ST-1006,Raúl Pinto,70.0,None


## 8. `DELETE`, `TRUNCATE` y `DROP`

Las tres "borran", pero con alcances muy distintos:

| Sentencia | Grupo | Qué elimina | ¿Admite `WHERE`? | En SQLite |
| --- | --- | --- | --- | --- |
| `DELETE FROM t WHERE ...` | DML | Las filas que cumplen la condición | Sí | Igual |
| `TRUNCATE TABLE t` | DDL | Todas las filas, más rápido; reinicia el contador `IDENTITY` | No | No existe: `DELETE FROM t` |
| `DROP TABLE t` | DDL | La tabla completa: estructura y datos | No | Igual |

In [9]:
conn.execute("CREATE TABLE Prueba (Id INTEGER)")
# random() devuelve un entero de 64 bits; abs(...) % 100 lo lleva a 0..99
conn.executemany("INSERT INTO Prueba VALUES (abs(random()) % 100)", [()] * 5)
print("Filas antes:", conn.execute("SELECT COUNT(*) FROM Prueba").fetchone()[0])

conn.execute("DELETE FROM Prueba")    # equivalente de TRUNCATE TABLE Prueba
print("Filas después de vaciarla:", conn.execute("SELECT COUNT(*) FROM Prueba").fetchone()[0])

conn.execute("DROP TABLE Prueba")
tablas = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()
print("Tablas que quedan:", sorted(t[0] for t in tablas))

Filas antes: 5
Filas después de vaciarla: 0
Tablas que quedan: ['Demo', 'Despacho']


## 9. `CASE`, `CAST` y concatenación

- `CASE WHEN condición THEN valor ... ELSE valor END` devuelve un valor según condiciones, como un `if/elif/else` dentro de la consulta.
- `CAST(expresión AS tipo)` convierte un valor a otro tipo.
- Para unir textos, SQLite usa `||`; T-SQL usa `+`. En SQLite, `+` es **siempre suma numérica**.

In [10]:
q("""
SELECT NumeroEnvio,
       PesoKg,
       CASE
           WHEN PesoKg < 2 THEN 'Ligero'
           WHEN PesoKg < 3.5 THEN 'Medio'
           ELSE 'Pesado'
       END                                   AS Categoria,
       'Cliente: ' || Cliente                AS Etiqueta,
       CAST(Costo AS INTEGER)                AS CostoEntero
FROM Despacho
""")

,NumeroEnvio,PesoKg,Categoria,Etiqueta,CostoEntero
0,EX-1001,2.5,Medio,Cliente: Ana Ruiz,198
1,ST-1002,1.2,Ligero,Cliente: Luis Mora,95
2,EX-1003,4.0,Pesado,Cliente: Marta Díaz,264
3,ST-1005,3.1,Medio,Cliente: Sofía Lara,120
4,ST-1006,1.0,Ligero,Cliente: Raúl Pinto,70


## 10. Errores frecuentes

**Olvidar el `WHERE` en `UPDATE` o `DELETE`.** La sentencia se ejecuta sin error y modifica toda la tabla. Dentro de una transacción todavía se puede deshacer con `rollback()`; después del `commit()`, no.

In [11]:
cur = conn.execute("UPDATE Despacho SET Cliente = 'Sin nombre'")   # sin WHERE
print("Filas modificadas:", cur.rowcount)
conn.rollback()   # se deshace porque aún no se confirmó
q("SELECT NumeroEnvio, Cliente FROM Despacho")

Filas modificadas: 5


,NumeroEnvio,Cliente
0,EX-1001,Ana Ruiz
1,ST-1002,Luis Mora
2,EX-1003,Marta Díaz
3,ST-1005,Sofía Lara
4,ST-1006,Raúl Pinto


**Concatenar con `+` en SQLite.** El código T-SQL `'EX-' + '1001'` no da error en SQLite, pero suma: convierte `'EX-'` a 0 y devuelve el número 1001. En T-SQL, además, `'EX-' + 1001` (texto más entero) falla al intentar convertir `'EX-'` a número; por eso allí se usa `CAST`.

In [12]:
q("SELECT 'EX-' + '1001' AS con_mas, 'EX-' || '1001' AS con_barras")

,con_mas,con_barras
0,1001,EX-1001


**Suponer que el tipo declarado valida los datos.** En SQLite, `INTEGER` o `VARCHAR(10)` no rechazan valores incorrectos. Si un dato debe cumplir una regla, se declara con `CHECK`, o se crea la tabla con la opción `STRICT`, que sí exige el tipo:

In [13]:
conn.execute("CREATE TABLE Medicion (Valor INTEGER) STRICT")
try:
    conn.execute("INSERT INTO Medicion VALUES ('alto')")
except sqlite3.IntegrityError as error:
    print("Rechazado:", error)

Rechazado: cannot store TEXT value in INTEGER column Medicion.Valor


**Depender de nombres de restricción autogenerados.** Un script que hace `DROP CONSTRAINT PK__Tabla__3A2F1B7C...` solo funciona en la base donde se generó ese nombre. Si la restricción se crea con `CONSTRAINT PK_Tabla PRIMARY KEY (...)`, el script es reproducible.

**Clave primaria de texto sin `NOT NULL` en SQLite.** Por compatibilidad histórica, SQLite permite `NULL` en una `PRIMARY KEY` que no sea `INTEGER`. Conviene declarar `NOT NULL` explícitamente, como en la tabla reconstruida.

## 11. Ejercicio

Una empresa registra a sus proveedores:

```python
proveedores = [
    (501, "Aceros del Norte", "MX", 4.5, "A"),
    (502, "Plásticos Andinos", "PE", 3.8, "I"),
    (503, "Tornillería Sur", "CL", 4.9, "A"),
    (504, "Embalajes Centro", "MX", 2.7, "A"),
    (505, "Química Austral", "AR", 3.2, "I"),
]
```

1. Crea la tabla `Proveedor(ProveedorID, Nombre, Pais, Calificacion, Estado)` con una clave primaria llamada `PK_Proveedor` y un `CHECK` llamado `CK_Proveedor_Estado` que solo admita `'A'` (activo) o `'I'` (inactivo). Inserta las filas en una sola sentencia.
2. Agrega la columna `Correo` y asigna `'contacto@aceros.mx'` al proveedor 501.
3. Reconstruye la tabla para que `ProveedorID` sea un texto con el país como prefijo (por ejemplo, `'MX-501'`), con la clave primaria `PK_Proveedor`.
4. Elimina los proveedores inactivos con calificación menor que 3.5 y muestra la tabla final con una columna `Nivel` que valga `'Preferente'` si la calificación es al menos 4.5 y `'Regular'` en otro caso.

### Solución

In [14]:
proveedores = [
    (501, "Aceros del Norte", "MX", 4.5, "A"),
    (502, "Plásticos Andinos", "PE", 3.8, "I"),
    (503, "Tornillería Sur", "CL", 4.9, "A"),
    (504, "Embalajes Centro", "MX", 2.7, "A"),
    (505, "Química Austral", "AR", 3.2, "I"),
]

conn.execute("""
CREATE TABLE Proveedor (
    ProveedorID  INTEGER NOT NULL,
    Nombre       TEXT    NOT NULL,
    Pais         TEXT,
    Calificacion REAL,
    Estado       TEXT,
    CONSTRAINT PK_Proveedor PRIMARY KEY (ProveedorID),
    CONSTRAINT CK_Proveedor_Estado CHECK (Estado IN ('A', 'I'))
)""")
# Una sola sentencia INSERT con todas las tuplas
filas = ", ".join(f"({i}, '{n}', '{p}', {c}, '{e}')" for i, n, p, c, e in proveedores)
conn.execute(f"INSERT INTO Proveedor (ProveedorID, Nombre, Pais, Calificacion, Estado) VALUES {filas}")

conn.execute("ALTER TABLE Proveedor ADD COLUMN Correo TEXT")
conn.execute("UPDATE Proveedor SET Correo = 'contacto@aceros.mx' WHERE ProveedorID = 501")
conn.commit()

q("SELECT * FROM Proveedor")

,ProveedorID,Nombre,Pais,Calificacion,Estado,Correo
0,501,Aceros del Norte,MX,4.5,A,contacto@aceros.mx
1,502,Plásticos Andinos,PE,3.8,I,None
2,503,Tornillería Sur,CL,4.9,A,None
3,504,Embalajes Centro,MX,2.7,A,None
4,505,Química Austral,AR,3.2,I,None


Los datos de este ejercicio son fijos y conocidos, por eso se arma el `INSERT` con un f-string. Con datos que provienen de usuarios se usan parámetros `?`, como se verá en el Módulo 40.

In [15]:
conn.executescript("""
BEGIN;
CREATE TABLE Proveedor_nuevo (
    ProveedorID  TEXT NOT NULL,
    Nombre       TEXT NOT NULL,
    Pais         TEXT,
    Calificacion REAL,
    Estado       TEXT,
    Correo       TEXT,
    CONSTRAINT PK_Proveedor PRIMARY KEY (ProveedorID),
    CONSTRAINT CK_Proveedor_Estado CHECK (Estado IN ('A', 'I'))
);
INSERT INTO Proveedor_nuevo
SELECT Pais || '-' || CAST(ProveedorID AS TEXT), Nombre, Pais, Calificacion, Estado, Correo
FROM Proveedor;
DROP TABLE Proveedor;
ALTER TABLE Proveedor_nuevo RENAME TO Proveedor;
COMMIT;
""")

cur = conn.execute("DELETE FROM Proveedor WHERE Estado = 'I' AND Calificacion < 3.5")
print("Proveedores eliminados:", cur.rowcount)
conn.commit()

q("""
SELECT ProveedorID, Nombre, Calificacion, Estado,
       CASE WHEN Calificacion >= 4.5 THEN 'Preferente' ELSE 'Regular' END AS Nivel
FROM Proveedor
""")

Proveedores eliminados: 1


,ProveedorID,Nombre,Calificacion,Estado,Nivel
0,MX-501,Aceros del Norte,4.5,A,Preferente
1,PE-502,Plásticos Andinos,3.8,I,Regular
2,CL-503,Tornillería Sur,4.9,A,Preferente
3,MX-504,Embalajes Centro,2.7,A,Regular


In [16]:
conn.close()

## Resumen

- El DDL (`CREATE`, `ALTER`, `DROP`, `TRUNCATE`) define la estructura; el DML (`INSERT`, `UPDATE`, `DELETE`) cambia las filas.
- SQLite usa afinidad de tipos y no valida longitudes: las reglas se imponen con `NOT NULL`, `CHECK`, `PRIMARY KEY` o tablas `STRICT`.
- Nombrar las restricciones (`CONSTRAINT PK_Tabla ...`) da mensajes de error claros y scripts que funcionan en cualquier instalación.
- Para cambiar el tipo de una columna o una restricción en SQLite se reconstruye la tabla: crear, copiar, borrar y renombrar, dentro de una transacción.
- `UPDATE` y `DELETE` sin `WHERE` afectan a toda la tabla; conviene probar antes el `WHERE` con un `SELECT`.
- Para concatenar se usa `||` en SQLite y `+` en T-SQL, con `CAST` cuando hay números.

En el **Módulo 37** se combinan varias tablas con `JOIN`, `UNION` y el manejo de valores nulos.